**Data loading and processing** 

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/merged/dyadic_panel_1992_2024_oda_capped_log.csv")

print(df.shape)
print(df.columns)
print(df[["sender_iso3", "recipient_iso3", "year"]].head())
print(df[["arms_tiv", "bilateral_oda", "econ_neocol_score_log", "colonial_tie"]].describe())
print(df[["arms_tiv", "bilateral_oda", "econ_neocol_score_log", "colonial_tie"]].isna().sum())

(115640, 15)
Index(['sender_iso3', 'recipient_iso3', 'year', 'arms_tiv', 'bilateral_oda',
       'econ_neocol_score', 'colonial_tie', 'journalist_killings',
       'gdp_per_capita', 'gdp_per_capita_log', 'population', 'population_log',
       'armed_conflict', 'conflict_intensity', 'econ_neocol_score_log'],
      dtype='object')
  sender_iso3 recipient_iso3  year
0         ABW            ISR  1996
1         AGO            CIV  2002
2         ALB            BFA  2011
3         ARE            BFA  2023
4         ARE            BGR  2021
            arms_tiv  bilateral_oda  econ_neocol_score_log   colonial_tie
count  115640.000000  103839.000000           80459.000000  115640.000000
mean        7.254507      17.930250               1.763019       0.037764
std        68.669320     122.492860               1.498681       0.190625
min         0.000000       0.000000               0.000000       0.000000
25%         0.000000       0.075692               0.470295       0.000000
50%         0.0

**Standardization**

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# ----------------------------
# 1.  missing values check again
# ----------------------------
df["arms_tiv"] = df["arms_tiv"].fillna(0)
df["bilateral_oda"] = df["bilateral_oda"].fillna(0)
df["econ_neocol_score"] = df["econ_neocol_score"].fillna(0)

# Zero-fill econ_neocol_score_log before use as edge weight.
# NaN → 0 means no edge is added (weight > 0 guard in compute_measures).
# This is correct: missing ECI data means no measurable asymmetry,
# not a zero-weight relationship. Produces identical network output
# to the implicit NaN filter, but makes the intent explicit.
# Documented limitations: ECI gap 1992–1994 leaves no econ edges for
# those years (uniform PageRank at 1/N); 68 non-ECI recipients similarly
# absent from the econ network layer. See LIMITATIONS.md.
df["econ_neocol_score_log"] = df["econ_neocol_score_log"].fillna(0)

# ----------------------------
# 2. Log-transformation
# ----------------------------
df["arms_tiv_weight"] = np.log1p(df["arms_tiv"])
df["bilateral_oda_weight"] = np.log1p(df["bilateral_oda"])
df["econ_neocol_score_weight"] = df["econ_neocol_score_log"]
df["colonial_tie_weight"] = df["colonial_tie"]


**Step 3: Runing the code to format it correctly and create the edge list**


In [4]:
edge_list = pd.concat(
    [
        df[["sender_iso3", "recipient_iso3", "year"]].assign(
            layer="arms_tiv",
            weight=df["arms_tiv_weight"]
        ),
        df[["sender_iso3", "recipient_iso3", "year"]].assign(
            layer="bilateral_oda",
            weight=df["bilateral_oda_weight"]
        ),
        df[["sender_iso3", "recipient_iso3", "year"]].assign(
            layer="econ_neocol_score",
            weight=df["econ_neocol_score_weight"]
        ),
        df[["sender_iso3", "recipient_iso3", "year"]].assign(
            layer="colonial_tie",
            weight=df["colonial_tie_weight"]
        )
    ],
    ignore_index=True
)


print(edge_list.shape)
edge_list.head()

(462560, 5)


,sender_iso3,recipient_iso3,year,layer,weight
0,ABW,ISR,1996,arms_tiv,2.933857
1,AGO,CIV,2002,arms_tiv,1.000632
2,ALB,BFA,2011,arms_tiv,0.788457
3,ARE,BFA,2023,arms_tiv,1.603420
4,ARE,BGR,2021,arms_tiv,0.542324


In [5]:
print(edge_list["layer"].value_counts())

edge_list.groupby("layer")["weight"].describe()

layer
arms_tiv             115640
bilateral_oda        115640
econ_neocol_score    115640
colonial_tie         115640
Name: count, dtype: int64


,count,mean,std,min,25%,50%,75%,max
layer,,,,,,,,
arms_tiv,115640.0,0.296889,1.004282,0.0,0.000000,0.000000,0.000000,8.267757
bilateral_oda,115640.0,1.122771,1.439488,0.0,0.029559,0.398288,1.888584,9.375096
colonial_tie,115640.0,0.037764,0.190625,0.0,0.000000,0.000000,0.000000,1.000000
econ_neocol_score,80459.0,1.763019,1.498681,0.0,0.470295,1.465783,2.741703,10.476703


**Network creation**

In [6]:
import networkx as nx

# ----------------------------
# Function to compute measures for one year & one layer
# ----------------------------
def compute_measures(df_subset, layer_name):
    """
    Compute network measures for one year and one layer.

    Input df_subset must contain:
    sender_iso3, recipient_iso3, year, layer, weight
    """

    G = nx.DiGraph()

    # Include all countries, even if they have no positive-weight edges
    nodes = set(df_subset["sender_iso3"]) | set(df_subset["recipient_iso3"])
    G.add_nodes_from(nodes)

    # Add only positive-weight edges
    # Zero-weight edges are treated as no observed tie
    for _, row in df_subset.iterrows():
        if row["weight"] > 0:
            G.add_edge(
                row["sender_iso3"],
                row["recipient_iso3"],
                weight=row["weight"]
            )

    # ----------------------------
    # 1. In-strength and out-strength
    # ----------------------------
    in_strength = dict(G.in_degree(weight="weight"))
    out_strength = dict(G.out_degree(weight="weight"))

    # ----------------------------
    # 2. PageRank
    # ----------------------------
    try:
        pagerank = nx.pagerank(G, weight="weight")
    except Exception as e:
        print(f"PageRank failed for layer {layer_name}: {e}")
        pagerank = {node: 0 for node in nodes}

    # ----------------------------
    # 3. Incoming concentration
    # ----------------------------
    # Measures whether incoming ties are dominated by one/few senders
    in_concentration = {}

    for node in nodes:
        incoming_edges = G.in_edges(node, data=True)
        weights = np.array([
            edge_data["weight"]
            for _, _, edge_data in incoming_edges
        ])

        if len(weights) == 0 or weights.sum() == 0:
            in_concentration[node] = 0
        else:
            shares = weights / weights.sum()
            in_concentration[node] = np.sum(shares ** 2)

    # ----------------------------
    # 4. Build output dataframe
    # ----------------------------
    data = []

    for node in nodes:
        i = in_strength.get(node, 0)
        o = out_strength.get(node, 0)

        data.append({
            "country": node,
            f"{layer_name}_in_strength": i,
            f"{layer_name}_out_strength": o,
            f"{layer_name}_dependency_balance": np.log1p(i) - np.log1p(o),
            f"{layer_name}_in_concentration": in_concentration.get(node, 0),
            f"{layer_name}_pagerank": pagerank.get(node, 0)
        })

    return pd.DataFrame(data)


# ----------------------------
# Loop over years and layers
# ----------------------------
results = []

years = sorted(edge_list["year"].unique())
layers = sorted(edge_list["layer"].unique())

for year in years:
    df_year = edge_list[edge_list["year"] == year]

    layer_dfs = []

    for layer in layers:
        df_layer = df_year[df_year["layer"] == layer]

        measures = compute_measures(df_layer, layer)
        layer_dfs.append(measures)

    # Merge all layer-measure datasets for this year
    df_merged = layer_dfs[0]

    for df_l in layer_dfs[1:]:
        df_merged = df_merged.merge(
            df_l,
            on="country",
            how="outer"
        )

    df_merged["year"] = year
    results.append(df_merged)


# ----------------------------
# Final network-measures dataset
# ----------------------------
network_measures = pd.concat(results, ignore_index=True)

# Countries with no activity in a layer get 0
network_measures = network_measures.fillna(0)

# ----------------------------
# Preview and sanity checks
# ----------------------------
print(network_measures.shape)
print(network_measures.head())
print(network_measures.isna().sum().sum())

(6526, 22)
  country  arms_tiv_in_strength  arms_tiv_out_strength  \
0     ABW                   0.0                    0.0   
1     AFG                   0.0                    0.0   
2     AGO                   0.0                    0.0   
3     AIA                   0.0                    0.0   
4     ALB                   0.0                    0.0   

   arms_tiv_dependency_balance  arms_tiv_in_concentration  arms_tiv_pagerank  \
0                          0.0                        0.0           0.003989   
1                          0.0                        0.0           0.003989   
2                          0.0                        0.0           0.003989   
3                          0.0                        0.0           0.003989   
4                          0.0                        0.0           0.003989   

   bilateral_oda_in_strength  bilateral_oda_out_strength  \
0                   3.429137                         0.0   
1                  25.434383           

In [7]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
network_measures.describe()

,arms_tiv_in_strength,arms_tiv_out_strength,arms_tiv_dependency_balance,arms_tiv_in_concentration,arms_tiv_pagerank,bilateral_oda_in_strength,bilateral_oda_out_strength,bilateral_oda_dependency_balance,bilateral_oda_in_concentration,bilateral_oda_pagerank,colonial_tie_in_strength,colonial_tie_out_strength,colonial_tie_dependency_balance,colonial_tie_in_concentration,colonial_tie_pagerank,econ_neocol_score_in_strength,econ_neocol_score_out_strength,econ_neocol_score_dependency_balance,econ_neocol_score_in_concentration,econ_neocol_score_pagerank,year
count,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000,6526.000000
mean,5.260849,5.260849,0.576874,0.303432,0.005057,19.895371,19.895371,1.504598,0.144778,0.005057,0.669169,0.669169,0.333250,0.495582,0.005057,21.736245,21.736245,1.191440,0.113140,0.005057,2007.792829
std,8.491217,20.527842,1.233651,0.366206,0.002683,21.218602,62.101933,2.842105,0.198925,0.001045,0.675202,4.649373,0.715046,0.470484,0.000368,24.446730,67.027347,2.573035,0.219396,0.001558,9.551601
min,0.000000,0.000000,-4.764087,0.000000,0.003455,0.000000,0.000000,-6.294788,0.000000,0.004220,0.000000,0.000000,-4.110874,0.000000,0.004634,0.000000,0.000000,-6.065281,0.000000,0.003926,1992.000000
25%,0.000000,0.000000,0.000000,0.000000,0.003932,1.220178,0.000000,0.320907,0.047190,0.004408,0.000000,0.000000,0.000000,0.000000,0.004907,0.000000,0.000000,0.000000,0.000000,0.004156,1999.000000
50%,1.214873,0.000000,0.000000,0.170016,0.004124,13.098600,0.000000,2.593942,0.083218,0.004727,1.000000,0.000000,0.693147,0.500000,0.004987,8.101768,0.000000,0.000000,0.051546,0.004679,2008.000000
75%,7.488589,0.000000,1.431936,0.511300,0.005026,32.383986,0.000000,3.503457,0.163878,0.005345,1.000000,0.000000,0.693147,1.000000,0.005099,43.506002,0.000000,3.769588,0.079894,0.005285,2016.000000
max,108.991998,243.521530,3.914911,1.000000,0.066294,164.356009,540.740734,5.108101,1.000000,0.022347,3.000000,60.000000,1.386294,1.000000,0.007204,101.367566,459.652111,4.628570,1.000000,0.023545,2024.000000


In [8]:
edge_list[edge_list["layer"] == "econ_neocol_score"].groupby("year")["weight"].agg(
    ["sum", "mean", "max"]
).head(15)

,sum,mean,max
year,,,
1992,0.000000,NaN,NaN
1993,0.000000,NaN,NaN
1994,0.000000,NaN,NaN
1995,4092.620478,2.113957,10.476703
1996,4046.614701,2.058298,9.776477
1997,4267.137918,2.014702,8.357861
1998,4346.670129,1.966819,9.188169
1999,4282.666558,1.944899,8.520406
2000,4458.801793,1.940297,8.498973


**Lag**

Ok so we need the lag because we want the network position to come before the journalist killing outcome.

So the model asks:

    Does a country’s neo-colonial network position last year help predict journalist killings this year?

In [9]:
# ----------------------------
# Create 1-year lagged network features
# ----------------------------

network_measures_lagged = network_measures.copy()

# Sort before lagging
network_measures_lagged = network_measures_lagged.sort_values(
    ["country", "year"]
)

# Identify network feature columns
feature_cols = [
    col for col in network_measures_lagged.columns
    if col not in ["country", "year"]
]

# Create lagged versions
for col in feature_cols:
    network_measures_lagged[f"{col}_lag1"] = (
        network_measures_lagged
        .groupby("country")[col]
        .shift(1)
    )

# Keep only country, year, and lagged features
lagged_cols = ["country", "year"] + [
    f"{col}_lag1" for col in feature_cols
]

network_features_lagged = network_measures_lagged[lagged_cols].copy()

# Preview
print(network_features_lagged.shape)
network_features_lagged.head()

(6526, 22)


,country,year,arms_tiv_in_strength_lag1,arms_tiv_out_strength_lag1,arms_tiv_dependency_balance_lag1,arms_tiv_in_concentration_lag1,arms_tiv_pagerank_lag1,bilateral_oda_in_strength_lag1,bilateral_oda_out_strength_lag1,bilateral_oda_dependency_balance_lag1,bilateral_oda_in_concentration_lag1,bilateral_oda_pagerank_lag1,colonial_tie_in_strength_lag1,colonial_tie_out_strength_lag1,colonial_tie_dependency_balance_lag1,colonial_tie_in_concentration_lag1,colonial_tie_pagerank_lag1,econ_neocol_score_in_strength_lag1,econ_neocol_score_out_strength_lag1,econ_neocol_score_dependency_balance_lag1,econ_neocol_score_in_concentration_lag1,econ_neocol_score_pagerank_lag1
0,ABW,1992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
205,ABW,1993,0.0,0.0,0.0,0.0,0.003989,3.429137,0.0,1.488205,1.0,0.004430,0.0,0.0,0.0,0.0,0.004721,0.0,0.0,0.0,0.0,0.004878
414,ABW,1994,0.0,0.0,0.0,0.0,0.003765,3.176803,0.0,1.429546,1.0,0.004371,0.0,0.0,0.0,0.0,0.004634,0.0,0.0,0.0,0.0,0.004785
622,ABW,1995,0.0,0.0,0.0,0.0,0.003786,2.829087,0.0,1.342626,1.0,0.004361,0.0,0.0,0.0,0.0,0.004655,0.0,0.0,0.0,0.0,0.004808
830,ABW,1996,0.0,0.0,0.0,0.0,0.003755,2.943386,0.0,1.372040,1.0,0.004389,0.0,0.0,0.0,0.0,0.004655,0.0,0.0,0.0,0.0,0.004126


In [10]:
# Rename join key to match monadic panel before saving
network_features_lagged = network_features_lagged.rename(columns={"country": "recipient_iso3"})

network_features_lagged.to_csv("../data/merged/network_measures_1992_2024.csv", index=False)
print(f"Saved: {network_features_lagged.shape}")
print(network_features_lagged.columns.tolist())
print(network_features_lagged.head(3))
print("\nMissing % per column:")
print((network_features_lagged.isna().mean() * 100).round(2).to_string())

Saved: (6526, 22)
['recipient_iso3', 'year', 'arms_tiv_in_strength_lag1', 'arms_tiv_out_strength_lag1', 'arms_tiv_dependency_balance_lag1', 'arms_tiv_in_concentration_lag1', 'arms_tiv_pagerank_lag1', 'bilateral_oda_in_strength_lag1', 'bilateral_oda_out_strength_lag1', 'bilateral_oda_dependency_balance_lag1', 'bilateral_oda_in_concentration_lag1', 'bilateral_oda_pagerank_lag1', 'colonial_tie_in_strength_lag1', 'colonial_tie_out_strength_lag1', 'colonial_tie_dependency_balance_lag1', 'colonial_tie_in_concentration_lag1', 'colonial_tie_pagerank_lag1', 'econ_neocol_score_in_strength_lag1', 'econ_neocol_score_out_strength_lag1', 'econ_neocol_score_dependency_balance_lag1', 'econ_neocol_score_in_concentration_lag1', 'econ_neocol_score_pagerank_lag1']
    recipient_iso3  year  arms_tiv_in_strength_lag1  \
0              ABW  1992                        NaN   
205            ABW  1993                        0.0   
414            ABW  1994                        0.0   

     arms_tiv_out_streng

In [11]:
network_features_lagged.isna().sum().sort_values(ascending=False).head(25)

bilateral_oda_pagerank_lag1                  215
colonial_tie_in_strength_lag1                215
econ_neocol_score_in_concentration_lag1      215
econ_neocol_score_dependency_balance_lag1    215
econ_neocol_score_out_strength_lag1          215
econ_neocol_score_in_strength_lag1           215
colonial_tie_pagerank_lag1                   215
colonial_tie_in_concentration_lag1           215
colonial_tie_dependency_balance_lag1         215
colonial_tie_out_strength_lag1               215
econ_neocol_score_pagerank_lag1              215
bilateral_oda_in_concentration_lag1          215
bilateral_oda_dependency_balance_lag1        215
bilateral_oda_out_strength_lag1              215
bilateral_oda_in_strength_lag1               215
arms_tiv_pagerank_lag1                       215
arms_tiv_in_concentration_lag1               215
arms_tiv_dependency_balance_lag1             215
arms_tiv_out_strength_lag1                   215
arms_tiv_in_strength_lag1                    215
year                

In [12]:
# ----------------------------
# Check final network outputs 
# ----------------------------

print("edge_list shape:")
print(edge_list.shape)
print(edge_list.head())

print("\nnetwork_measures shape:")
print(network_measures.shape)
print(network_measures.head())

print("\nnetwork_features_lagged shape:")
print(network_features_lagged.shape)
print(network_features_lagged.head())

edge_list shape:
(462560, 5)
  sender_iso3 recipient_iso3  year     layer    weight
0         ABW            ISR  1996  arms_tiv  2.933857
1         AGO            CIV  2002  arms_tiv  1.000632
2         ALB            BFA  2011  arms_tiv  0.788457
3         ARE            BFA  2023  arms_tiv  1.603420
4         ARE            BGR  2021  arms_tiv  0.542324

network_measures shape:
(6526, 22)
  country  arms_tiv_in_strength  arms_tiv_out_strength  \
0     ABW                   0.0                    0.0   
1     AFG                   0.0                    0.0   
2     AGO                   0.0                    0.0   
3     AIA                   0.0                    0.0   
4     ALB                   0.0                    0.0   

   arms_tiv_dependency_balance  arms_tiv_in_concentration  arms_tiv_pagerank  \
0                          0.0                        0.0           0.003989   
1                          0.0                        0.0           0.003989   
2               

In [13]:
print("edge_list columns:")
print(edge_list.columns.tolist())

print("\nnetwork_measures columns:")
print(network_measures.columns.tolist())

print("\nnetwork_features_lagged columns:")
print(network_features_lagged.columns.tolist())

edge_list columns:
['sender_iso3', 'recipient_iso3', 'year', 'layer', 'weight']

network_measures columns:
['country', 'arms_tiv_in_strength', 'arms_tiv_out_strength', 'arms_tiv_dependency_balance', 'arms_tiv_in_concentration', 'arms_tiv_pagerank', 'bilateral_oda_in_strength', 'bilateral_oda_out_strength', 'bilateral_oda_dependency_balance', 'bilateral_oda_in_concentration', 'bilateral_oda_pagerank', 'colonial_tie_in_strength', 'colonial_tie_out_strength', 'colonial_tie_dependency_balance', 'colonial_tie_in_concentration', 'colonial_tie_pagerank', 'econ_neocol_score_in_strength', 'econ_neocol_score_out_strength', 'econ_neocol_score_dependency_balance', 'econ_neocol_score_in_concentration', 'econ_neocol_score_pagerank', 'year']

network_features_lagged columns:
['recipient_iso3', 'year', 'arms_tiv_in_strength_lag1', 'arms_tiv_out_strength_lag1', 'arms_tiv_dependency_balance_lag1', 'arms_tiv_in_concentration_lag1', 'arms_tiv_pagerank_lag1', 'bilateral_oda_in_strength_lag1', 'bilateral_od